# PHARVO-beta: POS Unit Change Test

**Objective:** Verify that an authorized staff user (`rafi`) can toggle through the segmented unit controls (`PC` $\rightarrow$ `Strip` $\rightarrow$ `Box`) for a medicine (`Brufen`) in the POS catalog, and add the medicine with the final selected unit (`Box`) to the cart.

### Test Sequence
- **Medicine:** `Brufen`
- **Unit Sequence:** `PC` $\rightarrow$ `Strip` $\rightarrow$ `Box`
- **Final Cart Unit:** `Box`

### Prerequisites
```bash
pip install selenium webdriver-manager
```
Ensure PHARVO frontend is running at `http://localhost:5173` and backend at `http://localhost:8000`.

In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- Configuration & Test Data ---
BASE_URL = "http://localhost:5173"
USERNAME = "rafi"
PASSWORD = "password"  # Replace with actual password
MEDICINE_SEARCH = "Brufen"
UNIT_SEQUENCE = ["PC", "Strip", "Box"]
FINAL_UNIT = "Box"

# Step 1: Open browser and maximize window
driver = webdriver.Chrome()
driver.maximize_window()

# Set explicit wait helper (up to 10 seconds)
wait = WebDriverWait(driver, 10)

try:
    print(f"[INFO] Starting Unit Change Test for '{MEDICINE_SEARCH}' ({' -> '.join(UNIT_SEQUENCE)})...")

    # Step 2: Open login page and sign in
    driver.get(f"{BASE_URL}/")

    username_field = wait.until(
        EC.visibility_of_element_located((By.ID, "username"))
    )
    password_field = wait.until(
        EC.visibility_of_element_located((By.ID, "password"))
    )

    username_field.clear()
    username_field.send_keys(USERNAME)

    password_field.clear()
    password_field.send_keys(PASSWORD)

    sign_in_button = wait.until(
        EC.element_to_be_clickable((By.ID, "sign-in-btn"))
    )
    sign_in_button.click()

    # Step 3: Navigate to POS / Sales module via sidebar
    pos_nav = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//aside//button[contains(., 'POS / Sales')]")
        )
    )
    pos_nav.click()

    wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//header//h1[contains(text(), 'POS / Sales')]")
        )
    )
    print("[INFO] Navigated to POS / Sales terminal.")

    # Step 4: Search for target medicine
    pos_search_input = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//input[@aria-label='Search medicine or brand' and contains(@class, 'pos-input')]")
        )
    )
    pos_search_input.clear()
    pos_search_input.send_keys(MEDICINE_SEARCH)

    target_row = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, f"//table[contains(@class, 'pos-table')]//tbody//tr[contains(@class, 'pos-row-tr') and .//*[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), '{MEDICINE_SEARCH.lower()}')]]")
        )
    )
    med_name = target_row.find_element(By.XPATH, ".//td[1]//span[1]").text
    print(f"[INFO] Found target medicine: '{med_name}'")

    # Step 5: Test Unit Toggle Sequence (PC -> Strip -> Box)
    for unit_name in UNIT_SEQUENCE:
        unit_btn = target_row.find_element(
            By.XPATH, f".//button[(contains(@class, 'pos-seg-btn') or contains(@class, 'pos-unit-btn')) and (text()='{unit_name}' or contains(., '{unit_name}'))]"
        )

        # Check if the unit is enabled/available for this product
        if not unit_btn.is_enabled():
            print(f"[WARN] Unit '{unit_name}' is not enabled for {med_name}.")
            continue

        unit_btn.click()
        time.sleep(0.3)  # Brief pause to verify toggle reaction

        # Assert unit button is now pressed (aria-pressed='true')
        is_pressed = unit_btn.get_attribute("aria-pressed")
        print(f"[INFO] Toggled unit to '{unit_name}' (aria-pressed={is_pressed}).")
        assert is_pressed == "true", f"Expected unit '{unit_name}' to be selected (aria-pressed=true)"

    # Step 6: With the final unit (Box) selected, click Add to Cart
    add_btn = target_row.find_element(By.XPATH, ".//button[contains(., 'Add')]")
    assert add_btn.is_enabled(), f"'Add' button is disabled for unit '{FINAL_UNIT}'"
    add_btn.click()
    print(f"[INFO] Added medicine with unit '{FINAL_UNIT}' to cart.")

    # Step 7: Verify Cart Panel item displays the chosen unit (Box)
    cart_item_row = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, f"//tr[contains(@class, 'pos-row') and .//*[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), '{MEDICINE_SEARCH.lower()}')]]")
        )
    )

    unit_pill = cart_item_row.find_element(By.XPATH, ".//td[1]//*[contains(@class, 'pos-pill')]").text
    box_total = cart_item_row.find_element(By.XPATH, ".//td[4]/div[1]").text

    print(f"[INFO] Cart row item pill: '{unit_pill}', Line total: '{box_total}'")

    # Step 8: Assert final unit in cart
    if FINAL_UNIT.lower() in unit_pill.lower():
        print(f"PASS: Unit change sequence verified successfully.")
        print(f"      - Sequence Tested: {' -> '.join(UNIT_SEQUENCE)}")
        print(f"      - Final Unit in Cart: '{unit_pill}'")
        print(f"      - Line Total: '{box_total}'")
    else:
        print(f"FAIL: Expected final unit '{FINAL_UNIT}' in cart pill, got '{unit_pill}'.")

except Exception as error:
    print(f"FAIL: Unit Change test encountered error: {error}")

finally:
    # Step 9: Close browser
    print("[INFO] Cleaning up and closing browser...")
    driver.quit()
